AIM: To implement a Reinforcement Learning model for playing Tic Tac Toe using Q-learning.

ALGORITHM:
1. Create Tic Tac Toe board
2. Define states (board positions)
3. Define actions (empty positions)
4. Initialize Q-table
5. Choose action (explore/exploit)
6. Give reward:
Win = +1
Lose = -1
Draw = 0
7. Update Q-values using learning rule
8. Repeat for many games (training)
9. Test trained model

In [ ]:
import numpy as np
import random

# Initialize board
board = [" " for _ in range(9)]

# Print board
def print_board():
    print(board[0], "|", board[1], "|", board[2])
    print("--+---+--")
    print(board[3], "|", board[4], "|", board[5])
    print("--+---+--")
    print(board[6], "|", board[7], "|", board[8])

# Check winner
def check_winner(b, player):
    win_conditions = [
        [0,1,2],[3,4,5],[6,7,8],
        [0,3,6],[1,4,7],[2,5,8],
        [0,4,8],[2,4,6]
    ]
    return any(all(b[i] == player for i in cond) for cond in win_conditions)

# Available moves
def available_moves(b):
    return [i for i in range(9) if b[i] == " "]

# Q-table (dictionary)
Q = {}

# Learning parameters
alpha = 0.1
gamma = 0.9
epsilon = 0.2

# Get state
def get_state(b):
    return tuple(b)

# Choose action
def choose_action(state, moves):
    if random.random() < epsilon:
        return random.choice(moves)
    else:
        q_values = [Q.get((state, m), 0) for m in moves]
        return moves[np.argmax(q_values)]

# Update Q-value
def update_q(state, action, reward, next_state):
    old_q = Q.get((state, action), 0)
    # If next_state is terminal (no more moves), future_q should be 0
    if not available_moves(list(next_state)) and not check_winner(list(next_state), "X") and not check_winner(list(next_state), "O"):
        future_q = 0
    else:
        future_q = max([Q.get((next_state, a), 0) for a in range(9)], default=0)
    Q[(state, action)] = old_q + alpha * (reward + gamma * future_q - old_q)

# Training
for episode in range(500):
    board = [" " for _ in range(9)]
    state = get_state(board)

    while True:
        moves = available_moves(board)
        if not moves:
            # This case should ideally be caught after a player's move if it leads to a draw
            break

        action = choose_action(state, moves)
        board[action] = "X"

        # After AI's move, check for win or draw immediately
        if check_winner(board, "X"):
            update_q(state, action, 1, get_state(board))
            break

        # Check for draw after AI's move
        if not available_moves(board):
            update_q(state, action, 0, get_state(board)) # Draw, reward 0 for AI's last move
            break

        # Opponent (random)
        opp_moves = available_moves(board)
        opp_move = random.choice(opp_moves)
        board[opp_move] = "O"

        if check_winner(board, "O"):
            update_q(state, action, -1, get_state(board))
            break

        # Check for draw after Opponent's move
        if not available_moves(board):
            update_q(state, action, 0, get_state(board)) # Draw, reward 0 for AI's last move (considering opponent made a draw)
            break

        next_state = get_state(board)
        update_q(state, action, 0, next_state)
        state = next_state

# Testing
board = [" " for _ in range(9)]
print("Final Game (AI vs Random):")

while True:
    print_board()
    moves = available_moves(board)
    if not moves:
        print("Draw!")
        break

    state = get_state(board)
    action = choose_action(state, moves)
    board[action] = "X"

    if check_winner(board, "X"):
        print_board()
        print("AI Wins!")
        break

    if not available_moves(board):
        print("Draw!")
        break

    opp_move = random.choice(available_moves(board))
    board[opp_move] = "O"

    if check_winner(board, "O"):
        print_board()
        print("Opponent Wins!")
        break

    # Check for draw after Opponent's move in testing phase
    if not available_moves(board):
        print("Draw!")
        break

Final Game (AI vs Random):
  |   |  
--+---+--
  |   |  
--+---+--
  |   |  
X |   |  
--+---+--
  |   |  
--+---+--
  | O |  
X | X |  
--+---+--
  |   |  
--+---+--
O | O |  
X | X | X
--+---+--
  |   |  
--+---+--
O | O |  
AI Wins!


The reinforcement learning agent improved its gameplay by learning from rewards.